In [147]:
# math librairies
import numpy as np
from scipy import ndimage

from plotly import express
import pandas               # used with plotly

# 2D image processing 
import cv2

# custom librairies
import imicpe
import imicpe.optim as optim
print(imicpe.__version__)

1.0.14


# **Initialisation**

## Chargement de la vérité terrain

In [148]:
# génération d'une vérité terrain 1D xbar
N = 100
dom  = np.arange(1,N+1)
xbar = np.zeros((N,),float)
xbar[25:50] =  np.sin(.25*dom[25:50])
xbar[60:70] = -1.
xbar[75:90] =  1.


## Création de la donnée dégradée

In [149]:
# création de la donnée z, dégradée par un bruit blanc gaussien d'écart-type sig
sig = 0.2
z = xbar + sig * np.random.randn(N)


## Fonction de coût

In [150]:
# attache aux données
def f(x):
    return np.linalg.norm(x - z)**2

def grad_f(x):
    return 2 * (x - z)


def E(x, z, lam):
    return np.sum((x - z)**2) + lam * np.sum(np.abs(x))

# opérateurs gradient et divergence (1D)
def grad(x):
    return x[1:] - x[:-1]

def div(p):
    d = np.zeros(len(p)+1)
    d[0] = -p[0]
    d[1:-1] = p[:-1] - p[1:]
    d[-1] = p[-1]
    return d

def grad2d(x):
    gx = np.zeros_like(x)
    gy = np.zeros_like(x)
    gx[:, :-1] = x[:, 1:] - x[:, :-1]
    gy[:-1, :] = x[1:, :] - x[:-1, :]
    return gx, gy

def div2d(px, py):
    d = np.zeros_like(px)
    d[:, :-1] -= px[:, :-1]
    d[:, 1:] += px[:, :-1]
    d[:-1, :] -= py[:-1, :]
    d[1:, :] += py[:-1, :]
    return d



In [151]:
# opérateur proximal de la norme l1
def proxl1(x, tau):
    return np.sign(x) * np.maximum(np.abs(x) - tau, 0)


# **Algorithme**

In [152]:
# paramètres du modèle
lam = 1
gamma = 0.4        # < 1/L avec L = 2
Niter = 100

x = np.zeros(N)
En = []

for k in range(Niter):
    # étape forward (gradient)
    y = x - gamma * grad_f(x)
    
    # étape backward (prox)
    x = proxl1(y, gamma * lam)
    
    # énergie
    En.append(f(x) + lam * np.linalg.norm(x, 1))

xhat = x


# paramètres TV (1D)
lam_tv = 0.15
tau = 0.25
Niter_tv = 200

p = np.zeros(N-1)

for k in range(Niter_tv):
    gradF = -grad(z - 0.5 * div(p))
    p = p - tau * gradF
    p = np.clip(p, -lam_tv, lam_tv)

xhat_tv = z - 0.5 * div(p)


## **Affichage des resultats**

In [153]:
# plot cost function
plt_cost = pandas.DataFrame({'x':np.arange(Niter), 'y':En, 'legend':'cost', 'type':'cost_function'})

fig = express.line(plt_cost,
              x='x', 
              y='y', 
              log_x=True, log_y=True,
              labels={'x':'itérations (logscale)','y':'E(x^k) (logscale)'},
              title='Évolution de la fonction de coût en fonction des itérations',
              width=800, height=350)
fig.show()



In [154]:
plt_xbar = pandas.DataFrame({'x':dom, 'y':xbar, 'legend':'xbar'})
plt_z    = pandas.DataFrame({'x':dom, 'y':z,    'legend':'z'})
plt_lasso= pandas.DataFrame({'x':dom, 'y':xhat, 'legend':'LASSO'})
plt_tv   = pandas.DataFrame({'x':dom, 'y':xhat_tv, 'legend':'TV'})

data = pandas.concat([plt_xbar, plt_z, plt_lasso, plt_tv])

fig = express.line(
    data,
    x='x',
    y='y',
    color='legend',
    line_dash='legend',
    labels={'x':'indices','y':''},
    title='Comparaison LASSO vs TV (1D)',
    width=900, height=450
)
fig.show()

In [155]:
xbar = cv2.imread('images/starfish.jpg', 0)
if xbar is None:
    # If file not found, create a synthetic test image
    xbar = np.random.rand(256, 256)
else:
    xbar = xbar / 255.0

sig = 0.05
z = xbar + sig * np.random.randn(*xbar.shape)

lam = 0.15
tau = 0.25
Niter = 200

px = np.zeros_like(xbar)
py = np.zeros_like(xbar)

for k in range(Niter):
    div_p = div2d(px, py)
    gx, gy = grad2d(z - 0.5 * div_p)

    px += tau * gx
    py += tau * gy

    norm = np.maximum(1, np.sqrt(px**2 + py**2) / lam)
    px /= norm
    py /= norm

xhat_tv = z - 0.5 * div2d(px, py)


In [156]:
# plot images
plt_img = np.array([xbar, z, xhat_tv]) 
legend   = ['xbar', 'z', 'xhat_tv']
nb_img_per_row = 3
fig = express.imshow(plt_img, color_continuous_scale='gray', range_color=[0,1], 
                    title='Résultats',
                    facet_col=0, facet_col_spacing=0, facet_col_wrap=nb_img_per_row,
                    width=700, height=400,
                    )
fig.update_layout(coloraxis_showscale=True)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
item_map={f'{i}':key for i, key in enumerate(legend)}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]]))
fig.show()